# Session 10: Hosting, cost and licences

Price one hosting option from your own quotes on Hosting Cost Model.xlsx and complete Licence Register.xlsx with the licence files you have read. The data colleague runs the team notebook to summarise the register, total three options in cost units and flag words in licence text.

**SOLUTION VERSION. Tutor only.** This is the team notebook. Leaders can follow it on the shared screen. Run the cells from the top. Each cell that starts with '# Your turn' is yours to complete. Run it, then run the check cell after it: the check tells you if you are right and what to fix.

In [ ]:
# Run this cell first. It puts the course data files next to your notebook.
import os, zipfile, urllib.request
DATA_ZIP_URL = ""   # your tutor may give you a link; leave empty to upload data.zip instead
if not os.path.exists("customers.csv"):
    if DATA_ZIP_URL:
        urllib.request.urlretrieve(DATA_ZIP_URL, "data.zip")
    else:
        from google.colab import files
        print("Choose the file data.zip from your computer")
        uploaded = files.upload()
        os.replace(list(uploaded)[0], "data.zip")   # works even if the browser renamed it to data (1).zip
    zipfile.ZipFile("data.zip").extractall(".")
print("Data ready:", sorted(f for f in os.listdir(".") if f.endswith(".csv")), "and the docs folder")

## Task 1. The register by licence type
Read `licence_register_sample.csv`. Store the number of tools of each licence type in a dictionary called `by_type`.

In [ ]:
import pandas as pd

reg = pd.read_csv('licence_register_sample.csv')
by_type = reg.licence_type.value_counts().to_dict()
print(by_type)

In [ ]:
assert by_type == {'permissive': 10, 'proprietary': 2, 'source-available': 1, 'archived': 1, 'copyleft': 1}, 'Count the rows for each licence_type, as a dictionary'
print('Task 1 correct')

## Task 2. Tools that need a check
Store the names of tools marked `Check` in `needs_check`, the names of tools whose type is not permissive in `not_permissive`, and the number of tools with no `checked_on` date in `unread`.

In [ ]:
needs_check = reg[reg.free_for_commercial_use == 'Check'].tool.tolist()
not_permissive = reg[reg.licence_type != 'permissive'].tool.tolist()
unread = int(reg.checked_on.isna().sum())
print(len(needs_check), not_permissive, unread)

In [ ]:
assert len(needs_check) == 9 and 'Docker Desktop' in needs_check, 'Nine tools are marked Check'
assert sorted(not_permissive) == ['Docker Desktop', 'Google Colab', 'Metabase open edition', 'MinIO', 'Soda Core 4'], 'Five tools are not permissive'
assert unread == 15, 'No licence file has been read yet, so all 15 dates are blank'
print('Task 2 correct')

## Task 3. Total cost of three options, in cost units
The values below are invented cost units, not currency. Finish `tco` so it returns the total over `years`: a `once` line counts once, a `month` line counts 12 times the years, a `year` line counts the years. It must raise `ValueError` if any line is `None`. Then store each option's total in `totals`.

In [ ]:
BASIS = dict(hardware='once', hosting='month', connectivity='month', licences='year',
             engineers='month', evaluation='once', support='year', training='once',
             backup='year', security='once', exit='once')
options = {
    'On-site': [600, 8, 5, 4, 30, 40, 12, 10, 10, 20, 8],
    'Local data centre': [450, 14, 6, 4, 24, 40, 10, 10, 8, 20, 12],
    'Regional cloud': [40, 30, 8, 4, 16, 40, 8, 10, 6, 25, 30],
}
options = {n: dict(zip(BASIS, v)) for n, v in options.items()}

def tco(lines, years=3):
    missing = [k for k in BASIS if lines.get(k) is None]
    if missing:
        raise ValueError(f'cost lines not filled in: {missing}')
    times = {'once': 1, 'month': 12 * years, 'year': years}
    return sum(lines[k] * times[BASIS[k]] for k in BASIS)

totals = {n: tco(l) for n, l in options.items()}
print(totals)

In [ ]:
assert totals == {'On-site': 2304, 'Local data centre': 2182, 'Regional cloud': 2143}, 'Check the multipliers: month lines count 12 x years, year lines count years'
try:
    tco({**options['On-site'], 'exit': None})
    raise AssertionError('tco should raise ValueError when a line is missing')
except ValueError:
    pass
print('Task 3 correct')

## Task 4. How much does the people line matter?
Multiply the `engineers` line of every option by 0.5, 1.0 and 1.5. For each multiplier store the name of the cheapest option in `cheapest`, keyed by the multiplier.

In [ ]:
cheapest = {}
for scale in (0.5, 1.0, 1.5):
    tot = {n: tco({**l, 'engineers': l['engineers'] * scale}) for n, l in options.items()}
    cheapest[scale] = min(tot, key=tot.get)
print(cheapest)

In [ ]:
assert cheapest == {0.5: 'Local data centre', 1.0: 'Regional cloud', 1.5: 'Regional cloud'}, 'Total every option at each multiplier and keep the name of the lowest'
print('Task 4 correct')

## Task 5. Flag words in licence texts
Finish `flag` so it returns the names from `KEYS` whose pattern appears in a text. Store the flags for each sample in `found`, and the sample names that return no flag in `missed`.

In [ ]:
import re

KEYS = {'AGPL': r'\bAGPL', 'Elastic License': r'Elastic License', 'SSPL': r'\bSSPL\b',
        'Apache': r'\bApache\b', 'MIT': r'\bMIT\b', 'Business Source': r'Business Source'}
samples = {
    'A': 'MIT License. Permission is hereby granted, free of charge, to any person.',
    'B': 'Elastic License 2.0. You may not provide the software as a hosted service. These limitations apply.',
    'C': 'Licensed under Apache License 2.0. The bundled console is under the AGPL v3.',
    'D': 'GNU Affero General Public License, version 3. You must offer the source to network users.',
}

def flag(text):
    return [k for k, pattern in KEYS.items() if re.search(pattern, text)]

found = {n: flag(t) for n, t in samples.items()}
missed = [n for n, f in found.items() if not f]
print(found, missed)

In [ ]:
assert found == {'A': ['MIT'], 'B': ['Elastic License'], 'C': ['AGPL', 'Apache'], 'D': []}, 'Use re.search with each pattern in KEYS'
assert missed == ['D'], 'Sample D names the AGPL in full, so no keyword catches it'
print('Task 5 correct')

## For the leader
Look at your results. Which of the five tools that are not permissive would you refuse to build on until its licence file has been read, and who in your organisation signs that decision? What is our exit if the vendor changes terms? Write your answer in two sentences below.